# Steering Toward Foldability: what if we aim at the thing we actually want?

## The question

Every steering experiment in this project so far injects a **repetition** vector -- built by
contrasting repetitive against non-repetitive sequences, because that is what PEP's UCCS method
targets. The finding is that it reduces repetition and destroys structure.

But there is an obvious experiment nobody has run: **build the vector from well-folding versus
badly-folding sequences instead, and push the model directly toward foldability.**

This is the exact mirror of `24-ai4dd-utility-matched-fair-test.ipynb`. There, D+/D- are split
by repetition and then *utility-matched* so foldability is not the difference. Here, D+/D- are
split by utility and then *repetition-matched* so repetition is not the difference. Same
candidate pool, same scoring functions, same normalization, same injection mechanism -- the only
thing that changes is which axis the contrast is built along.

## Why this is worth a session

Both outcomes are strong, which is the property to look for when GPU time is scarce.

**If it helps** -- collapse drops below CONTROL at some strength -- that is a genuine positive
result and a materially better paper. It would mean activation steering on protein LMs is not
inherently destructive; PEP's method fails because *repetition is the wrong target*, not because
the technique is broken. That reframes the whole contribution from "here is a cautionary
negative result" to "here is why the published method fails, and here is the fix."

**If it does not help** -- collapse rises with strength exactly as the repetition vector does --
that is the strongest available statement of this project's central claim: *even steering
directly toward the property you want destroys it.* Magnitude does not just dominate direction
among arbitrary directions; it dominates even the maximally favourable one. That single sentence
is worth more to the paper than another model replication.

## The design choice that decides whether this is informative

Everything in this project so far shows nothing measurable happens at 1x and severe collapse
happens at 2x. If any *benefit* from a foldability direction exists, it will live **below** 1x --
at a push small enough not to be swamped by magnitude damage. Testing only 1x and 2x would very
likely produce a null and tell us nothing about the direction itself.

So this notebook deliberately tests **0.25x, 0.5x, 1x, 2x**: a ladder weighted toward the low
end, where a directional benefit could actually show up, plus 2x to confirm the damage regime
still behaves as expected. This is the one place in the project where the gentle conditions are
the interesting ones.

## Reference points

- Same candidate-pool recipe as `24` (N=200 real ProtGPT2 generations, all folded), so this
  run's CONTROL and pool statistics should land near `24`'s (natural collapse 53.0%, CONTROL
  62.0%). If they do not, treat that as a run-to-run sampling issue and compare within this
  notebook only.
- `24`'s locked repetition-vector numbers at the same layer, norm and multiplier
  (`notes/locked-results.md` §1g): CONTROL 62.0% -> L12_FAIR_1x 60.0% -> L12_FAIR_2x 90.0%.
  The foldability vector is built from the *same pool in the same run*, so the two vectors are
  directly comparable at 1x and 2x.
- Residual-stream norm ‖h‖ is measured here too, so results can be reported in `alpha_rel`
  units per `34-ai4dd-residual-norm-audit.ipynb` and §1k-FLAG.

Kaggle setup: Accelerator = **GPU T4 x1 or x2**, Internet = **ON**. Expect ~2-2.5 hours
(200 candidate folds + 250 condition folds).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding

torch.manual_seed(2024)
np.random.seed(2024)

device = "cuda" if torch.cuda.is_available() else "cpu"

REFERENCE_NORM = 583.998
TARGET_LAYER = 12
N_CANDIDATES = 200
N_PER_CONDITION = 50

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())


Setup complete. CUDA available: True


In [2]:
# --- Identical scoring functions to 24/26/27/28/32, unchanged. R(x) higher = less repetitive;
#     U(x) higher = folds better. This notebook swaps which of the two is used to SPLIT the
#     contrastive sets and which is used to MATCH them -- nothing about the formulas changes. ---

ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0
        inputs = self.tokenizer([cleaned], return_tensors="pt", add_special_tokens=False).to(self.device)
        plddt, ptm = 0.0, 0.0
        try:
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
        except RuntimeError:
            clear_gpu()
        return plddt, ptm

def fold_records_ptm(records, evaluator):
    for r in records:
        plddt, ptm = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["collapse"] = int(0.0 < plddt < 60.0)
        r["repetition_score"] = repetition_score(r.get("gen_only", r["sequence"]))
        r["utility_score"] = utility_score(plddt, ptm) if plddt > 0 else 0.0
    return records

print("Scoring functions and ESMFold evaluator ready.")


Scoring functions and ESMFold evaluator ready.


In [3]:
# --- Candidate pool: identical construction to 24, so the pool this vector is built from is
#     the same kind of object as the one the repetition vector was built from. ---
UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

# Offline fallback -- the same real protein fragments hardcoded in
# 03-ai4dd-uccs-baseline-test.ipynb (its positive_seqs). Real sequences already committed to
# this repo and already used to build a locked result; nothing here is synthetic.
FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print()
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING.")
    print("!!")
    print("!! A 'name resolution' error here means Kaggle's Internet toggle is OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!!")
    print("!! With Internet off the HuggingFace downloads will also fail unless cached.")
    print("!! Turning Internet on is the real fix. This fallback only prevents a crash.")
    print("!!")
    print("!! WARNING specific to this notebook: the whole experiment depends on generating a")
    print("!! DIVERSE 200-candidate pool with real spread in foldability, since D+/D- are")
    print("!! selected from its extremes. Six source proteins instead of twelve narrows that")
    print("!! spread and may weaken the contrast the vector is built from. Prefer re-running")
    print("!! with Internet ON rather than interpreting a fallback run.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=11):
    if not reference_seqs:
        raise RuntimeError(
            "No reference sequences available -- neither the UniProt fetch nor the fallback "
            "produced anything. Check Kaggle's Internet setting before re-running."
        )
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

candidate_prefixes = build_prefix_pool(reference_seqs, n_prefixes=N_CANDIDATES)
print(f"Built {len(candidate_prefixes)} candidate prefixes.")

print(f"Loading ProtGPT2 on {device}...")
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

def generate_natural(tokenizer, model, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    records = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=tokenizer.eos_token_id
            )
        seq = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_only = seq[len(prompt):] if seq.startswith(prompt) else seq
        records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_only})
    clear_gpu()
    return records

print(f"=== Generating N={N_CANDIDATES} natural candidate sequences ===")
candidate_records = generate_natural(tokenizer, plm_model, candidate_prefixes, max_len=50, seed=505)
print(f"Generated {len(candidate_records)} candidates.")

print("=== Freeing ProtGPT2 while ESMFold folds the pool ===")
del plm_model
clear_gpu()

evaluator = StructuralEvaluatorPTM()
print("Folding and scoring the candidate pool...")
candidate_records = fold_records_ptm(candidate_records, evaluator)
del evaluator
clear_gpu()

valid_candidates = [r for r in candidate_records if r["plddt"] > 0.0]
print(f"\n{len(valid_candidates)}/{len(candidate_records)} candidates folded successfully.")
print(f"Repetition score range: {min(r['repetition_score'] for r in valid_candidates):.3f} - "
      f"{max(r['repetition_score'] for r in valid_candidates):.3f}")
print(f"Utility score range:    {min(r['utility_score'] for r in valid_candidates):.3f} - "
      f"{max(r['utility_score'] for r in valid_candidates):.3f}")
print(f"Mean pLDDT: {np.mean([r['plddt'] for r in valid_candidates]):.2f}, "
      f"natural collapse rate: {np.mean([r['collapse'] for r in valid_candidates]):.1%}")
print(f"(24 recorded: pool collapse 53.0%, mean pLDDT 58.13 -- close values here confirm the")
print(f" pool is comparable and the two vectors can be read against each other.)")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues
Built 200 candidate prefixes.
Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding and scoring the candidate pool...

200/200 candidates folded successfully.
Repetition score range: 0.340 - 0.980
Utility score range:    0.153 - 0.827
Mean pLDDT: 57.43, natural collapse rate: 57.0%
(24 recorded: pool collapse 53.0%, mean pLDDT 58.13 -- close values here confirm the
 pool is comparable and the two vectors can be read against each other.)


In [4]:
# --- THE MIRROR OF FIX 1. 24 splits on repetition and matches on utility. This splits on
#     UTILITY and matches on REPETITION -- so the one systematic difference between D+ and D-
#     is how well they fold, not how repetitive they are.
#
#     Without the repetition-matching step this vector would be confounded in exactly the way
#     24 was built to avoid: badly-folding sequences also tend to be repetitive, so an unmatched
#     "foldability" vector would partly just be the repetition vector again, and a null result
#     would be uninterpretable. ---

QUANTILE = 0.30
REPETITION_TOLERANCE = 0.05   # epsilon on mean R(x), mirroring 24's 0.05 on mean U(x)

sorted_by_utility = sorted(valid_candidates, key=lambda r: r["utility_score"])
n_side = max(10, int(len(sorted_by_utility) * QUANTILE))

d_lowfold_raw = sorted_by_utility[:n_side]     # worst-folding  (low U)
d_highfold_raw = sorted_by_utility[-n_side:]   # best-folding   (high U)

mean_r_high_before = np.mean([r["repetition_score"] for r in d_highfold_raw])
mean_r_low_before = np.mean([r["repetition_score"] for r in d_lowfold_raw])
print(f"Before repetition-matching: D+(well-folding) mean R={mean_r_high_before:.3f} "
      f"(n={len(d_highfold_raw)}), D-(badly-folding) mean R={mean_r_low_before:.3f} "
      f"(n={len(d_lowfold_raw)}), gap={abs(mean_r_high_before - mean_r_low_before):.3f}")

def match_on(pool_a, pool_b, key, tolerance, max_iters=200):
    # Trim the higher-mean side one extreme member at a time until the two pools' mean `key`
    # values are within `tolerance`. Same algorithm as 24's utility_match, parameterised by
    # which score is being equalised.
    a, b = list(pool_a), list(pool_b)
    for _ in range(max_iters):
        mean_a = np.mean([r[key] for r in a])
        mean_b = np.mean([r[key] for r in b])
        gap = mean_a - mean_b
        if abs(gap) <= tolerance or min(len(a), len(b)) <= 15:
            break
        if gap > 0:
            a.sort(key=lambda r: -r[key])
            a.pop(0)
        else:
            b.sort(key=lambda r: r[key])
            b.pop(0)
    return a, b

d_highfold, d_lowfold = match_on(d_highfold_raw, d_lowfold_raw,
                                 key="repetition_score", tolerance=REPETITION_TOLERANCE)

mean_r_high = np.mean([r["repetition_score"] for r in d_highfold])
mean_r_low = np.mean([r["repetition_score"] for r in d_lowfold])
mean_u_high = np.mean([r["utility_score"] for r in d_highfold])
mean_u_low = np.mean([r["utility_score"] for r in d_lowfold])

print(f"\nAfter repetition-matching:")
print(f"  D+ (well-folding) : n={len(d_highfold):3d}  mean U(x)={mean_u_high:.3f}  mean R(x)={mean_r_high:.3f}")
print(f"  D- (badly-folding): n={len(d_lowfold):3d}  mean U(x)={mean_u_low:.3f}  mean R(x)={mean_r_low:.3f}")
print(f"  Utility separation  (Delta U): {mean_u_high - mean_u_low:.3f}   <- the signal we want")
print(f"  Residual repetition gap (Delta R): {abs(mean_r_high - mean_r_low):.3f} "
      f"(target <= {REPETITION_TOLERANCE})   <- the confound we removed")

if abs(mean_r_high - mean_r_low) > REPETITION_TOLERANCE:
    print("\nWARNING: repetition gap did not close to tolerance. The resulting vector is partly")
    print("a repetition vector, and a null result would be ambiguous. Report the residual gap")
    print("explicitly if this happens, and treat the run as weaker evidence than intended.")
if (mean_u_high - mean_u_low) < 0.10:
    print("\nWARNING: utility separation is small after matching -- the vector may be too weak a")
    print("contrast to test the hypothesis properly. Note this alongside any null result.")


Before repetition-matching: D+(well-folding) mean R=0.902 (n=60), D-(badly-folding) mean R=0.781 (n=60), gap=0.121

After repetition-matching:
  D+ (well-folding) : n= 18  mean U(x)=0.621  mean R(x)=0.827
  D- (badly-folding): n= 60  mean U(x)=0.278  mean R(x)=0.781
  Utility separation  (Delta U): 0.343   <- the signal we want
  Residual repetition gap (Delta R): 0.046 (target <= 0.05)   <- the confound we removed


In [5]:
# --- Build the foldability vector, and (free, from the same pool) the repetition vector, so
#     the two directions can be compared to each other directly rather than across notebooks. ---

print(f"Reloading ProtGPT2 on {device} to extract activations...")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

def get_mean_activation(model, tokenizer, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

# The foldability direction: mean(well-folding) - mean(badly-folding).
pos_acts = get_mean_activation(plm_model, tokenizer, [r["sequence"] for r in d_highfold], TARGET_LAYER)
neg_acts = get_mean_activation(plm_model, tokenizer, [r["sequence"] for r in d_lowfold], TARGET_LAYER)
v_fold_raw = pos_acts.mean(dim=0) - neg_acts.mean(dim=0)
v_fold_rawnorm = v_fold_raw.norm().item()
v_fold = (v_fold_raw * (REFERENCE_NORM / v_fold_rawnorm)).to(device)
print(f"Foldability vector: raw norm = {v_fold_rawnorm:.4f} -> normalized to {v_fold.norm().item():.4f}")

# The repetition direction from the SAME pool, for a direct cosine comparison. Built with 24's
# own convention (split on repetition, match on utility) so it is the same object 24 tested.
sorted_by_rep = sorted(valid_candidates, key=lambda r: r["repetition_score"])
dm_raw = sorted_by_rep[:n_side]
dp_raw = sorted_by_rep[-n_side:]
d_plus, d_minus = match_on(dp_raw, dm_raw, key="utility_score", tolerance=0.05)
pos_r = get_mean_activation(plm_model, tokenizer, [r["sequence"] for r in d_plus], TARGET_LAYER)
neg_r = get_mean_activation(plm_model, tokenizer, [r["sequence"] for r in d_minus], TARGET_LAYER)
v_rep_raw = pos_r.mean(dim=0) - neg_r.mean(dim=0)
v_rep = (v_rep_raw * (REFERENCE_NORM / v_rep_raw.norm().item())).to(device)
print(f"Repetition vector (same pool, 24's recipe): raw norm = {v_rep_raw.norm().item():.4f} "
      f"-> normalized to {v_rep.norm().item():.4f}")

cos = float(torch.dot(v_fold.cpu().float(), v_rep.cpu().float()) /
            (v_fold.cpu().float().norm() * v_rep.cpu().float().norm()))
print(f"\nCosine similarity between the foldability and repetition directions: {cos:+.4f}")
print("  ~0     -> genuinely different directions; this is a real independent test.")
print("  ~+1    -> they are effectively the same direction, and a null result here says nothing")
print("            new (report this honestly -- it would mean repetition and foldability are not")
print("            separable in this model's activation space, which is itself a finding).")
print("  ~-1    -> the foldability direction is the repetition direction reversed.")

# Residual-stream norm at the injection site, for alpha_rel reporting.
def measure_resid_norm(model, tokenizer, seqs, layer):
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        captured["h"] = h.detach()
    handle = model.transformer.h[layer].register_forward_hook(hook)
    vals = []
    try:
        for seq in seqs:
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
            captured.clear()
            with torch.no_grad():
                model(**inputs)
            if "h" in captured:
                vals.append(captured["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(vals)) if vals else float("nan")

resid_norm = measure_resid_norm(plm_model, tokenizer,
                                [r["sequence"] for r in valid_candidates[:40]], TARGET_LAYER)
alpha_rel_1x = REFERENCE_NORM / resid_norm
print(f"\nMean residual-stream norm ‖h‖ at layer {TARGET_LAYER}: {resid_norm:.2f}")
print(f"  => alpha_rel at 1x = {alpha_rel_1x:.4f}")


Reloading ProtGPT2 on cuda to extract activations...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Foldability vector: raw norm = 584.8858 -> normalized to 583.9981
Repetition vector (same pool, 24's recipe): raw norm = 328.8438 -> normalized to 583.9980

Cosine similarity between the foldability and repetition directions: +0.9819
  ~0     -> genuinely different directions; this is a real independent test.
  ~+1    -> they are effectively the same direction, and a null result here says nothing
            new (report this honestly -- it would mean repetition and foldability are not
            separable in this model's activation space, which is itself a finding).
  ~-1    -> the foldability direction is the repetition direction reversed.

Mean residual-stream norm ‖h‖ at layer 12: 2892.82
  => alpha_rel at 1x = 0.2019


In [6]:
# --- The dose ladder, deliberately weighted toward gentle pushes. If steering toward
#     foldability ever helps, it helps here -- below the magnitude regime where everything
#     collapses regardless of direction. ---

def generate_with_vector_steering(model, tokenizer, target_layer, steering_vector, prompts,
                                  max_len=50, seed=0):
    torch.manual_seed(seed)
    model.eval()
    v = None if steering_vector is None else steering_vector.to(device)

    def hook(module, inp, out):
        if v is None:
            return out
        return (out[0] + v,)

    records = []
    for prompt in prompts:
        handle = model.transformer.h[target_layer].register_forward_hook(hook)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=tokenizer.eos_token_id
            )
        handle.remove()
        seq = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_part = seq[len(prompt):] if seq.startswith(prompt) else seq
        records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_part,
                        "entropy": calculate_entropy(gen_part)})
    clear_gpu()
    return records

steer_prefixes = build_prefix_pool(reference_seqs, n_prefixes=300, seed=23)

CONDITION_SPECS = [
    ("CONTROL",     None,          0.0, 111),
    ("FOLD_0.25x",  v_fold * 0.25, 0.25, 222),
    ("FOLD_0.5x",   v_fold * 0.5,  0.5,  333),
    ("FOLD_1x",     v_fold * 1.0,  1.0,  444),
    ("FOLD_2x",     v_fold * 2.0,  2.0,  555),
]

conditions = {}
multipliers = {}
for i, (name, vec, mult, seed) in enumerate(CONDITION_SPECS):
    lo = i * N_PER_CONDITION
    hi = lo + N_PER_CONDITION
    print(f"=== {name} (prefixes {lo}:{hi}, seed {seed}) ===")
    conditions[name] = generate_with_vector_steering(
        plm_model, tokenizer, TARGET_LAYER, vec, steer_prefixes[lo:hi], seed=seed)
    multipliers[name] = mult

print("\n=== Freeing ProtGPT2 from GPU ===")
del plm_model
clear_gpu()

evaluator2 = StructuralEvaluatorPTM()
for name in conditions:
    print(f"Folding {name}...")
    for r in conditions[name]:
        plddt, ptm = evaluator2.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["collapse"] = int(0.0 < plddt < 60.0)
del evaluator2
clear_gpu()

print(f"\n{'Condition':14s} {'N':>4s} {'Entropy':>9s} {'pLDDT':>8s} {'pTM':>7s} {'Collapse%':>10s}")
print("-" * 58)
for name, recs in conditions.items():
    ents = [r["entropy"] for r in recs]
    plddts = [r["plddt"] for r in recs if r["plddt"] > 0.0]
    ptms = [r["ptm"] for r in recs if r["plddt"] > 0.0]
    print(f"{name:14s} {len(recs):4d} {np.mean(ents):9.3f} {np.mean(plddts):8.2f} "
          f"{np.mean(ptms):7.3f} {np.mean([r['collapse'] for r in recs]) * 100:9.1f}%")


=== CONTROL (prefixes 0:50, seed 111) ===
=== FOLD_0.25x (prefixes 50:100, seed 222) ===
=== FOLD_0.5x (prefixes 100:150, seed 333) ===
=== FOLD_1x (prefixes 150:200, seed 444) ===
=== FOLD_2x (prefixes 200:250, seed 555) ===

=== Freeing ProtGPT2 from GPU ===
Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL...
Folding FOLD_0.25x...
Folding FOLD_0.5x...
Folding FOLD_1x...
Folding FOLD_2x...

Condition         N   Entropy    pLDDT     pTM  Collapse%
----------------------------------------------------------
CONTROL          50     2.847    59.17   0.296      52.0%
FOLD_0.25x       50     2.581    62.07   0.286      46.0%
FOLD_0.5x        50     2.820    60.33   0.311      46.0%
FOLD_1x          50     2.449    58.95   0.263      62.0%
FOLD_2x          50     2.751    45.49   0.141      90.0%


In [7]:
# --- Analysis. The question is directional: does ANY dose beat CONTROL? ---
from scipy.stats import fisher_exact

def wilson_ci(k, n, z=1.959963985):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

summary = {}
for name, recs in conditions.items():
    k = int(np.sum([r["collapse"] for r in recs]))
    n = len(recs)
    summary[name] = {"k": k, "n": n, "rate": k / n, "ci": wilson_ci(k, n),
                     "plddt": float(np.mean([r["plddt"] for r in recs if r["plddt"] > 0])),
                     "mult": multipliers[name]}

ctrl = summary["CONTROL"]

print("=" * 96)
print("DOES STEERING TOWARD FOLDABILITY HELP?")
print("=" * 96)
print(f"{'Condition':14s} {'collapsed':>12s} {'rate':>8s} {'95% CI':>20s} {'alpha_rel':>10s} "
      f"{'vs CONTROL p':>13s} {'dir':>6s}")
print("-" * 96)
for name in conditions:
    s = summary[name]
    ci_str = "[{:.1%}, {:.1%}]".format(s["ci"][0], s["ci"][1])
    if name == "CONTROL":
        p_str, direction = "--", "--"
    else:
        _, p = fisher_exact([[s["k"], s["n"] - s["k"]], [ctrl["k"], ctrl["n"] - ctrl["k"]]])
        p_str = f"{p:.4f}"
        direction = "BETTER" if s["rate"] < ctrl["rate"] else ("worse" if s["rate"] > ctrl["rate"] else "same")
    print(f"{name:14s} {s['k']:6d}/{s['n']:<5d} {s['rate']:7.1%} {ci_str:>20s} "
          f"{s['mult'] * alpha_rel_1x:10.4f} {p_str:>13s} {direction:>6s}")

print()
print("Reference -- 24's repetition vector, same layer/norm/multiplier (locked §1g):")
print("  CONTROL 62.0%  ->  L12_FAIR_1x 60.0%  ->  L12_FAIR_2x 90.0%")
print(f"This run's foldability vector:")
print(f"  CONTROL {summary['CONTROL']['rate']:.1%}  ->  FOLD_1x {summary['FOLD_1x']['rate']:.1%}"
      f"  ->  FOLD_2x {summary['FOLD_2x']['rate']:.1%}")
print(f"Cosine(foldability, repetition) in this run: {cos:+.4f}")

print()
print("=" * 96)
print("VERDICT")
print("=" * 96)

best_name = min((n for n in conditions if n != "CONTROL"), key=lambda n: summary[n]["rate"])
best = summary[best_name]
_, p_best = fisher_exact([[best["k"], best["n"] - best["k"]], [ctrl["k"], ctrl["n"] - ctrl["k"]]])
worst_name = max((n for n in conditions if n != "CONTROL"), key=lambda n: summary[n]["rate"])
monotone_up = all(summary[a]["rate"] <= summary[b]["rate"] + 1e-9 for a, b in
                  zip(["CONTROL", "FOLD_0.25x", "FOLD_0.5x", "FOLD_1x"],
                      ["FOLD_0.25x", "FOLD_0.5x", "FOLD_1x", "FOLD_2x"]))

print(f"Best condition: {best_name} at {best['rate']:.1%} vs CONTROL {ctrl['rate']:.1%} "
      f"(Fisher p = {p_best:.4f})")
print(f"Worst condition: {worst_name} at {summary[worst_name]['rate']:.1%}")
print(f"Collapse rises monotonically with dose: {monotone_up}")
print()

if best["rate"] < ctrl["rate"] and p_best < 0.05:
    print("  ==> STEERING TOWARD FOLDABILITY HELPS. This is a positive result and it changes the")
    print("      paper's thesis: activation steering on protein LMs is not inherently")
    print("      destructive, and PEP's method fails because repetition is the wrong target --")
    print("      not because the technique cannot work. Lead the Results section with this.")
    print("      Follow-up worth doing if time allows: confirm at a second layer, and check")
    print("      whether the helped sequences are also less repetitive (i.e. whether you get")
    print("      PEP's intended benefit for free) or whether it is a pure foldability gain.")
elif best["rate"] < ctrl["rate"]:
    print("  ==> A NUMERICAL IMPROVEMENT THAT DOES NOT CLEAR SIGNIFICANCE at N=50. Do not claim")
    print("      steering toward foldability helps. Report it as a non-significant trend and, if")
    print("      GPU time remains, re-run the single best dose at N=150-200 -- this is the one")
    print("      result in the project where a scale-up could convert a hedge into a headline.")
else:
    print("  ==> STEERING TOWARD FOLDABILITY DOES NOT HELP -- collapse never drops below CONTROL")
    print("      at any dose, including the gentle ones where a directional benefit had the best")
    print("      chance to appear.")
    print()
    print("      This is the strongest available statement of the paper's central claim:")
    print("      *even steering directly toward the property you want destroys it.* Magnitude")
    print("      dominates not merely over arbitrary directions but over the maximally")
    print("      favourable one. Pair this with 35-ai4dd-random-direction-control.ipynb --")
    print("      together they bracket the claim from both sides (a random direction hurts as")
    print("      much as the real one; the ideal direction helps no more than none).")


DOES STEERING TOWARD FOLDABILITY HELP?
Condition         collapsed     rate               95% CI  alpha_rel  vs CONTROL p    dir
------------------------------------------------------------------------------------------------
CONTROL            26/50      52.0%       [38.5%, 65.2%]     0.0000            --     --
FOLD_0.25x         23/50      46.0%       [33.0%, 59.6%]     0.0505        0.6893 BETTER
FOLD_0.5x          23/50      46.0%       [33.0%, 59.6%]     0.1009        0.6893 BETTER
FOLD_1x            31/50      62.0%       [48.2%, 74.1%]     0.2019        0.4193  worse
FOLD_2x            45/50      90.0%       [78.6%, 95.7%]     0.4038        0.0000  worse

Reference -- 24's repetition vector, same layer/norm/multiplier (locked §1g):
  CONTROL 62.0%  ->  L12_FAIR_1x 60.0%  ->  L12_FAIR_2x 90.0%
This run's foldability vector:
  CONTROL 52.0%  ->  FOLD_1x 62.0%  ->  FOLD_2x 90.0%
Cosine(foldability, repetition) in this run: +0.9819

VERDICT
Best condition: FOLD_0.25x at 46.0% vs CO

In [8]:
# --- Persist. Same CSV discipline as 35. ---
rows = []
for name, recs in conditions.items():
    for i, r in enumerate(recs):
        rows.append({
            "condition": name, "idx": i, "multiplier": multipliers[name],
            "alpha_rel": multipliers[name] * alpha_rel_1x,
            "prompt": r["prompt"], "sequence": r["sequence"], "gen_only": r["gen_only"],
            "gen_length": len(r["gen_only"]),
            "usable_length": sum(1 for a in r["gen_only"] if a in VALID_AA),
            "entropy": r["entropy"], "plddt": r["plddt"], "ptm": r["ptm"],
            "collapse": r["collapse"],
        })
pd.DataFrame(rows).to_csv("foldability_steering_sequences.csv", index=False)

pool_rows = [{
    "sequence": r["sequence"], "gen_only": r["gen_only"],
    "plddt": r["plddt"], "ptm": r["ptm"], "collapse": r["collapse"],
    "repetition_score": r["repetition_score"], "utility_score": r["utility_score"],
} for r in valid_candidates]
pd.DataFrame(pool_rows).to_csv("foldability_candidate_pool.csv", index=False)

pd.DataFrame([{
    "condition": n, "collapsed": s["k"], "n": s["n"], "collapse_rate": s["rate"],
    "ci_lo": s["ci"][0], "ci_hi": s["ci"][1], "mean_plddt": s["plddt"],
    "multiplier": s["mult"], "alpha_rel": s["mult"] * alpha_rel_1x,
} for n, s in summary.items()]).to_csv("foldability_steering_summary.csv", index=False)

print("Saved:")
print("  foldability_steering_sequences.csv  (every generated sequence, all conditions)")
print("  foldability_candidate_pool.csv      (the 200-candidate pool with R/U scores)")
print("  foldability_steering_summary.csv    (per-condition collapse rates + CIs)")
print()
print("The candidate-pool CSV is reusable well beyond this notebook -- it is the first saved")
print("record in this project of natural ProtGPT2 output with folding scores attached, which is")
print("what the failure-mode taxonomy, length-confound check and ESM-2 second-opinion analyses")
print("all need (see notes/proposed-extensions.md).")


Saved:
  foldability_steering_sequences.csv  (every generated sequence, all conditions)
  foldability_candidate_pool.csv      (the 200-candidate pool with R/U scores)
  foldability_steering_summary.csv    (per-condition collapse rates + CIs)

The candidate-pool CSV is reusable well beyond this notebook -- it is the first saved
record in this project of natural ProtGPT2 output with folding scores attached, which is
what the failure-mode taxonomy, length-confound check and ESM-2 second-opinion analyses
all need (see notes/proposed-extensions.md).
